In [10]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ============================================================
# 1. test 데이터 불러오기
# ============================================================

test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_test_fe.parquet')

print(test.shape)
test.head()

(186263, 33)


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,imerg_rain_24h,sfc_rain_1h,sfc_rain_6h,sfc_rain_24h,radar_reflectivity_mean,radar_reflectivity_p95,radar_reflectivity_ge20_fraction,cat_area,wsarea,mean_elevation,mean_slope,urban_ratio,forest_ratio,stream_order,wl_diff_1h,wl_diff_3h,wl_diff_6h,wl_diff_12h,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h
row_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
TE_000102207,2100-01-11,ST0001,0.12,0.12,0.13,0.14,0.16,0.06,0.00,0.000,0.507,0.0,0.0,2.5,5.544,9.930,0.0000,2.8605,37.0279,124.37,9.6147,11.1141,52.4597,1.9592,0.00,-0.01,-0.02,-0.04,0.06,0.0000,0.028167,0.0,0.138889
TE_000068213,2100-01-11,ST0002,0.82,0.83,0.84,0.85,0.85,0.86,0.00,0.000,1.939,0.0,0.0,0.5,10.777,25.907,0.1113,3.9564,42.8163,180.09,16.5687,2.5710,88.0622,1.8000,-0.01,-0.02,-0.03,-0.03,-0.04,0.0000,0.107722,0.0,0.027778
TE_000126042,2100-01-11,ST0003,0.79,0.79,0.79,0.78,0.78,0.78,0.00,0.000,0.135,0.0,0.0,3.5,8.932,14.590,0.0000,4.2240,80.8559,125.04,12.1864,34.3816,56.6211,2.2500,0.00,0.00,0.01,0.01,0.01,0.0000,0.007500,0.0,0.194444
TE_000111041,2100-01-11,ST0005,1.01,1.01,1.02,1.05,1.07,0.98,0.00,0.000,0.678,0.0,0.0,6.0,5.930,12.030,0.0000,3.6470,118.5237,246.72,17.1248,0.4779,88.0002,2.0000,0.00,-0.01,-0.04,-0.06,0.03,0.0000,0.037667,0.0,0.333333
TE_000129309,2100-01-11,ST0006,0.67,0.65,0.63,0.67,0.80,0.70,0.19,1.251,2.005,0.5,0.5,2.0,21.713,30.330,0.6045,11.8928,14.9987,38.11,5.7671,63.7535,18.7559,1.2222,0.02,0.04,0.00,-0.13,-0.03,0.2122,0.041889,0.0,0.083333


In [13]:
# 미래의 1~6h wl_t와 차분해서 미래수위변동 컬럼 추가하기

# 필요한 컬럼만 추출 (row_id 인덱스 유지)
test_future = test[['timestamp', 'station_id', 'wl_t']].copy()

# 관측소와 시각 기준 수위 조회용 시리즈(row_id가 빠지지만 원본이 아니므로 상관없음)
wl_lookup = test_future.set_index(['station_id', 'timestamp'])['wl_t']

# 1~6시간 후 수위 상승량 계산
for h in range(1, 7):
    future_index = pd.MultiIndex.from_arrays([  # 멀티인덱스 객체를 생성해라
        test_future['station_id'],  # 첫번째 인덱스
        test_future['timestamp'] + pd.Timedelta(hours=h)  # 두번째 인덱스는 시간+1~6시간 순회하면서 생성
    ])

    test_future[f'+{h}h_future_rise'] = (  # h_future_rise라는 컬럼을 생성
        wl_lookup.reindex(future_index).to_numpy()  # 위에서 받은 future_index에 해당하는 값을 가져오고(없으면 NaN)
        - test_future['wl_t'].to_numpy()  # 그 행의 현재수위를 차분
    )
    # 넘파이배열로 변경 이유: 인덱스를 떼기위해서(인덱스가 있으면 인덱스끼리 연산이 일어남)
display(test_future.head())

print('전체 행 개수: ', len(test_future))

future_rise_cols = ['+1h_future_rise',
                    '+2h_future_rise', 
                    '+3h_future_rise', 
                    '+4h_future_rise', 
                    '+5h_future_rise',
                    '+6h_future_rise']
print(
    'future_rise에 NaN이 하나도 없는 행 수: ',
    (~test_future[future_rise_cols].isnull().any(axis=1)).sum()
)

,timestamp,station_id,wl_t,+1h_future_rise,+2h_future_rise,+3h_future_rise,+4h_future_rise,+5h_future_rise,+6h_future_rise
row_id,,,,,,,,,
TE_000102207,2100-01-11,ST0001,0.12,-0.01,-0.01,NaN,NaN,NaN,NaN
TE_000068213,2100-01-11,ST0002,0.82,-0.01,-0.02,NaN,NaN,NaN,NaN
TE_000126042,2100-01-11,ST0003,0.79,0.01,0.00,NaN,NaN,NaN,NaN
TE_000111041,2100-01-11,ST0005,1.01,-0.01,0.02,NaN,NaN,NaN,NaN
TE_000129309,2100-01-11,ST0006,0.67,0.02,0.03,NaN,NaN,NaN,NaN


전체 행 개수:  186263
future_rise에 NaN이 하나도 없는 행 수:  114111


In [ ]:
# 6h 최대상승량 컬럼 만들기

test_future['future_maxrise'] = (
    test_future[future_rise_cols]
    .max(axis=1, skipna=False)
)

test_future = test_future.dropna(subset=['future_maxrise'])

print('총 행열 수: ', test_future.shape)
display(test_future.head())

총 행열 수:  (114111, 10)


,timestamp,station_id,wl_t,+1h_future_rise,+2h_future_rise,+3h_future_rise,+4h_future_rise,+5h_future_rise,+6h_future_rise,future_maxrise
row_id,,,,,,,,,,
TE_000046543,2100-03-12,ST0001,0.19,-0.01,-0.02,-0.02,-0.02,-0.01,-0.01,-0.01
TE_000111013,2100-03-12,ST0002,0.85,-0.01,-0.01,0.00,0.01,0.01,0.01,0.01
TE_000086280,2100-03-12,ST0003,0.76,0.00,0.00,0.00,0.00,0.01,0.01,0.01
TE_000136298,2100-03-12,ST0005,0.98,0.00,0.01,0.01,0.01,0.02,0.02,0.02
TE_000166154,2100-03-12,ST0006,0.73,0.02,0.02,0.01,0.00,-0.01,-0.03,0.02


In [20]:
# 기존 test셋에 합치기
# print(test.index.name)
# print(test_future.index.name)

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

test[target_col] = test_future['future_maxrise']
test = test.dropna(subset=target_col)

print(test.shape)

(114111, 34)


In [21]:
# parquet 저장
test.to_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

In [1]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [6]:
# ============================================================
# Test 정답 복원 여부에 따른 독립변수 분포 비교
# ============================================================

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 데이터 불러오기

from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

test_all = pd.read_parquet(
    PROJECT_ROOT / 'parqueted_data' / '1_test_fe.parquet'
)

test_recovered = pd.read_parquet(
    PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet'
)

# 복원 여부 구분
recovered_mask = test_all.index.isin(test_recovered.index)

print('전체 Test:', len(test_all))
print('복원:', recovered_mask.sum())
print('미복원:', (~recovered_mask).sum())
print('복원 비율:', round(recovered_mask.mean() * 100, 2), '%')

# 분위수 비교
quantiles = [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1]

dist_recovered = (
    test_all.loc[recovered_mask, num_cols_fe]
    .quantile(quantiles)
    .T
)

dist_missing = (
    test_all.loc[~recovered_mask, num_cols_fe]
    .quantile(quantiles)
    .T
)

# 기존 분위수 결과를 세로로 재배치
dist_compare = pd.concat(
    {
        '복원': dist_recovered.stack(),
        '미복원': dist_missing.stack()
    },
    axis=1
)

dist_compare['차이'] = (
    dist_compare['복원'] - dist_compare['미복원']
)

dist_compare.index.names = ['독립변수', '분위수']

display(dist_compare.round(4))

전체 Test: 186263
복원: 114111
미복원: 72152
복원 비율: 61.26 %


복원         미복원       차이
독립변수                             분위수                                  
wl_t                             0.00     -5.1800     -5.1400  -0.0400
                                 0.01     -1.0000     -1.0600   0.0600
                                 0.05     -0.1050     -0.1200   0.0150
                                 0.25      0.6400      0.5800   0.0600
                                 0.50      1.1800      1.0900   0.0900
                                 0.75      2.0900      1.9300   0.1600
                                 0.95     25.0400     25.0345   0.0055
                                 0.99     80.9040     82.2096  -1.3056
                                 1.00    137.2600    139.7000  -2.4400
wl_t_minus_1h                    0.00     -5.1800     -5.1500  -0.0300
                                 0.01     -1.0000     -1.0600   0.0600
                                 0.05     -0.1100     -0.1200   0.0100
                                 0.25      0.6300      0.5700   0.0600
                                 0.50      1.1800      1.0800   0.1000
                                 0.75      2.0800      1.9200   0.1600
                                 0.95     25.0400     25.0445  -0.0045
                                 0.99     80.9020     82.1747  -1.2727
                                 1.00    137.2600    139.6800  -2.4200
wl_t_minus_3h                    0.00     -5.1900     -5.1600  -0.0300
                                 0.01     -1.0000     -1.0700   0.0700
                                 0.05     -0.1200     -0.1200   0.0000
                                 0.25      0.6200      0.5600   0.0600
                                 0.50      1.1600      1.0700   0.0900
                                 0.75      2.0500      1.9100   0.1400
                                 0.95     25.0400     25.0500  -0.0100
                                 0.99     80.8990     82.1098  -1.2108
                                 1.00    137.2500    139.6800  -2.4300
wl_t_minus_6h                    0.00     -5.1900     -5.1700  -0.0200
                                 0.01     -1.0100     -1.0500   0.0400
                                 0.05     -0.1200     -0.1300   0.0100
                                 0.25      0.6100      0.5500   0.0600
                                 0.50      1.1400      1.0600   0.0800
                                 0.75      2.0200      1.9000   0.1200
                                 0.95     25.0400     25.0600  -0.0200
                                 0.99     80.8930     82.0398  -1.1468
                                 1.00    137.2500    139.6600  -2.4100
wl_t_minus_12h                   0.00     -5.2000     -5.1900  -0.0100
                                 0.01     -1.0400     -1.0700   0.0300
                                 0.05     -0.1400     -0.1400   0.0000
                                 0.25      0.5900      0.5500   0.0400
                                 0.50      1.1000      1.0500   0.0500
                                 0.75      1.9500      1.8900   0.0600
                                 0.95     25.0500     25.0500   0.0000
                                 0.99     80.8370     81.9800  -1.1430
                                 1.00    137.3200    139.6700  -2.3500
wl_t_minus_24h                   0.00     -5.1900     -5.1900   0.0000
                                 0.01     -1.0500     -1.0800   0.0300
                                 0.05     -0.1500     -0.1400  -0.0100
                                 0.25      0.5700      0.5300   0.0400
                                 0.50      1.0700      1.0300   0.0400
                                 0.75      1.8800      1.8600   0.0200
                                 0.95     25.0500     25.0600  -0.0100
                                 0.99     80.8410     81.9349  -1.0939
                                 1.00    137.3500    139.6400  -2.2900
imerg_rain_1h                    0.00      0.0000      0.0000   0.0000
